Dependencies

In [1]:
%pip install -q openai anthropic pydantic python-dotenv pandas openpyxl


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

PROVIDER = os.getenv("PROVIDER", "openai")

The use story

In [3]:
STORY = """US-214: As a customer, I want to log in with my mobile number and a one-time password (OTP),
so that I don't need to remember a password.

Acceptance criteria:
AC1: Mobile number must be exactly 10 digits and start with 6, 7, 8 or 9.
AC2: A 6-digit OTP is sent by SMS when the user taps 'Send OTP'.
AC3: The OTP is valid for 5 minutes from the time it is sent.
AC4: After 3 wrong OTP attempts, login is locked for 30 minutes.
AC5: 'Resend OTP' is enabled only 30 seconds after the last OTP was sent. At most 3 resends per session.
AC6: On the correct OTP, the user lands on the home screen and stays logged in for 30 days on that device."""

CRITERIA = ["AC1", "AC2", "AC3", "AC4", "AC5", "AC6"]

In [4]:
MUST_TEST = [
    ("Mobile with 9 digits",               ["9 digit", "9-digit", "nine digit"]),
    ("Mobile with 11 digits",              ["11 digit", "11-digit", "eleven digit"]),
    ("Mobile starting with 0-5",           ["starts with 5", "starting with 5", "starts with 0", "starting with 0",
                                            "first digit 5", "first digit 0", "begins with 5", "begins with 0"]),
    ("OTP with 5 or 7 digits",             ["5 digit", "5-digit", "7 digit", "7-digit"]),
    ("OTP used at 4:59 (just valid)",      ["4:59", "4 minutes 59", "299 sec", "just before expir"]),
    ("OTP used at 5:01 (just expired)",    ["5:01", "5 minutes 1 ", "301 sec", "just after expir", "after expiry"]),
    ("3rd wrong attempt locks",            ["3rd wrong", "third wrong", "3 wrong", "three wrong"]),
    ("Resend tapped at 29 seconds",        ["29 sec", "29s", "before 30 sec"]),
    ("4th resend is blocked",              ["4th resend", "fourth resend", "more than 3 resend", "exceed"]),
]
print(len(MUST_TEST), "must-test boundaries")

9 must-test boundaries


Test case schema

In [5]:
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError

class TestCase(BaseModel):
    id: str = Field(pattern=r"^TC-\d{3}$", description="TC-001, TC-002, ...")
    title: str = Field(max_length=100, description="What is being checked, in one line")
    type: Literal["positive", "negative", "boundary", "edge"]
    priority: Literal["High", "Medium", "Low"]
    criterion: str = Field(pattern=r"^AC\d+$", description="Which acceptance criterion this test checks")
    preconditions: list[str] = Field(default_factory=list)
    steps: list[str] = Field(min_length=1, description="One action per step")
    test_data: Optional[str] = Field(None, description="Exact values to use, e.g. mobile 98765 4321")
    expected_result: str = Field(description="What the tester should see. Must be checkable.")

class TestSuite(BaseModel):
    story_id: str
    test_cases: list[TestCase] = Field(min_length=5)

In [6]:
# The ID pattern and the type list reject sloppy output:
try:
    TestCase(id="TC1", title="Login works", type="happy path", priority="High",
             criterion="AC6", steps=["Log in"], expected_result="It works")
except ValidationError as e:
    print(e)

2 validation errors for TestCase
id
  String should match pattern '^TC-\d{3}$' [type=string_pattern_mismatch, input_value='TC1', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_pattern_mismatch
type
  Input should be 'positive', 'negative', 'boundary' or 'edge' [type=literal_error, input_value='happy path', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


Helpers

In [7]:
import json, re

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-haiku-4-5-20251001")

def call_llm(system: str, user: str, temperature: float = 0.2) -> str:
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=OPENAI_MODEL, temperature=temperature,
            response_format={"type": "json_object"},
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}])
        return resp.choices[0].message.content
    if PROVIDER == "anthropic":
        import anthropic
        resp = anthropic.Anthropic().messages.create(
            model=CLAUDE_MODEL, max_tokens=8000, temperature=temperature,
            system=system, messages=[{"role": "user", "content": user}])
        return resp.content[0].text
    raise RuntimeError(f"Unknown PROVIDER {PROVIDER}, must be openai or anthropic")

def extract_json(text: str) -> str:
    start, end = text.find("{"), text.rfind("}")
    return text[start:end + 1] if start != -1 and end != -1 else text

def ask_json(system, user, model_cls, temperature=0.2, max_retries=2):
    """Same helper as Projects 1 and 2."""
    error = None
    for attempt in range(max_retries + 1):
        prompt = user if error is None else (
            f"{user}\n\nYour previous JSON failed validation:\n{error}\nReturn corrected JSON only.")
        raw = call_llm(system, prompt, temperature)
        try:
            return model_cls.model_validate_json(extract_json(raw))
        except ValidationError as e:
            error = str(e)
            print(f"attempt {attempt + 1} failed validation, retrying")
    raise RuntimeError(f"Failed after {max_retries + 1} attempts:\n{error}")

Zero shot

In [8]:
SYSTEM_V1 = f"""Write test cases for this user story.
Return ONLY a JSON object matching this schema:
{json.dumps(TestSuite.model_json_schema(), indent=2)}"""

v1 = ask_json(SYSTEM_V1, STORY, TestSuite)
print(len(v1.test_cases), "test cases")
for t in v1.test_cases:
    print(f"{t.id} [{t.type:8}] {t.criterion}  {t.title}")

18 test cases
TC-001 [positive] AC1  Login with valid mobile and correct OTP within validity
TC-002 [negative] AC1  Reject mobile number shorter than 10 digits
TC-003 [boundary] AC1  Reject mobile number longer than 10 digits
TC-004 [negative] AC1  Reject mobile number starting with invalid digit
TC-005 [boundary] AC1  Accept mobile numbers starting with 6, 7, 8, or 9
TC-006 [positive] AC2  Send 6-digit OTP when tapping Send OTP
TC-007 [negative] AC2  Do not send OTP when mobile number is invalid
TC-008 [boundary] AC3  OTP expires after 5 minutes from being sent
TC-009 [boundary] AC3  OTP is accepted just before 5-minute expiry
TC-010 [positive] AC4  Lock login after 3 consecutive wrong OTP attempts
TC-011 [edge    ] AC4  Prevent login attempts during 30-minute lockout
TC-012 [boundary] AC4  Allow login after 30-minute lockout period ends
TC-013 [boundary] AC5  Resend OTP button disabled for first 30 seconds
TC-014 [boundary] AC5  Enable Resend OTP after 30 seconds
TC-015 [edge    ] AC

Coverage - measure the test suite in code

In [9]:
import difflib
import pandas as pd

def tc_text(t: TestCase) -> str:
    return " ".join([t.title, t.test_data or "", " ".join(t.steps), t.expected_result]).lower()

def coverage(suite: TestSuite) -> dict:
    tcs = suite.test_cases
    acs = {t.criterion for t in tcs}
    types = {t.type for t in tcs}
    hit = [name for name, kws in MUST_TEST if any(any(k in tc_text(t) for k in kws) for t in tcs)]
    titles = [t.title.lower() for t in tcs]
    dupes = sum(1 for i in range(len(titles)) for j in range(i + 1, len(titles))
                if difflib.SequenceMatcher(None, titles[i], titles[j]).ratio() > 0.85)
    vague = sum(1 for t in tcs if re.search(r"works|as expected|correctly|successfully|follows the", t.expected_result.lower()))
    return {"tests": len(tcs),
            "criteria covered": f"{len(acs & set(CRITERIA))}/{len(CRITERIA)}",
            "test types": f"{len(types)}/4",
            "boundaries hit": f"{len(hit)}/{len(MUST_TEST)}",
            "near-duplicates": dupes,
            "vague expected results": vague,
            "_missing": [n for n, _ in MUST_TEST if n not in hit]}

def show_coverage(suite):
    c = coverage(suite)
    print({k: v for k, v in c.items() if not k.startswith("_")})
    print("Missing:", c["_missing"] or "nothing")

show_coverage(v1)

{'tests': 18, 'criteria covered': '6/6', 'test types': '4/4', 'boundaries hit': '4/9', 'near-duplicates': 1, 'vague expected results': 2}
Missing: ['OTP with 5 or 7 digits', 'OTP used at 4:59 (just valid)', 'OTP used at 5:01 (just expired)', 'Resend tapped at 29 seconds', '4th resend is blocked']


Domain specific prompt

In [10]:
QA_RULES = """You are a senior QA engineer. Design tests the way an expert would:
1. Equivalence partitioning: split each input into valid and invalid groups; test one value from each group.
2. Boundary value analysis: for every limit in the criteria (lengths, counts, times), test just below,
   exactly at, and just above the limit.
3. Negative tests: wrong format, empty input, special characters, wrong order of actions.
4. Edge cases: network loss, app killed mid-flow, two devices, time-zone or clock changes.
5. Every test links to exactly one acceptance criterion and checks ONE thing.
6. Expected results must be observable: an exact message, a screen, or a state. Never "works fine".
7. Put exact values in test_data, e.g. "9876543210", "OTP entered at 4:59".
8. Priority: High = money, security or login blocked; Medium = wrong but recoverable; Low = cosmetic.
9. Do not repeat the same test with trivially different data."""

SYSTEM_V2 = f"""{QA_RULES}
Return ONLY a JSON object matching this schema:
{json.dumps(TestSuite.model_json_schema(), indent=2)}"""

v2 = ask_json(SYSTEM_V2, STORY, TestSuite)
show_coverage(v2)

{'tests': 21, 'criteria covered': '6/6', 'test types': '4/4', 'boundaries hit': '2/9', 'near-duplicates': 1, 'vague expected results': 0}
Missing: ['Mobile with 9 digits', 'Mobile with 11 digits', 'Mobile starting with 0-5', 'OTP with 5 or 7 digits', 'OTP used at 4:59 (just valid)', 'Resend tapped at 29 seconds', '4th resend is blocked']


few shot

In [11]:
# One worked example from a DIFFERENT feature (checkout coupons), to teach format and boundary thinking.
EXAMPLE_STORY = """US-101: As a shopper, I want to apply a coupon code at checkout.
AC1: Coupon code is 6 to 10 letters or digits.
AC2: Coupon applies only when the cart total is at least Rs 500."""

EXAMPLE_TESTS = {"story_id": "US-101", "test_cases": [
  {"id": "TC-001", "title": "Valid 6-character coupon on a Rs 750 cart", "type": "positive", "priority": "High",
   "criterion": "AC1", "preconditions": ["Cart total Rs 750"], "steps": ["Enter SAVE10", "Tap Apply"],
   "test_data": "SAVE10", "expected_result": "Discount line appears and total is reduced"},
  {"id": "TC-002", "title": "5-character coupon is rejected (just below minimum length)", "type": "boundary",
   "priority": "Medium", "criterion": "AC1", "preconditions": ["Cart total Rs 750"],
   "steps": ["Enter SAVE1", "Tap Apply"], "test_data": "SAVE1 (5 characters)",
   "expected_result": "Error 'Enter a valid coupon code'; total unchanged"},
  {"id": "TC-003", "title": "Cart of Rs 499 cannot use a coupon (just below threshold)", "type": "boundary",
   "priority": "High", "criterion": "AC2", "preconditions": ["Cart total Rs 499"],
   "steps": ["Enter SAVE10", "Tap Apply"], "test_data": "Cart Rs 499",
   "expected_result": "Message 'Add Rs 1 more to use this coupon'; total unchanged"}]}

SYSTEM_V3 = f"""{QA_RULES}

Example story:
{EXAMPLE_STORY}

Example tests (an extract; a full suite would have more):
{json.dumps(EXAMPLE_TESTS, indent=2)}

Now write a FULL suite for the real story. Copy the style and the boundary thinking, not the content.
Return ONLY a JSON object matching this schema:
{json.dumps(TestSuite.model_json_schema(), indent=2)}"""

v3 = ask_json(SYSTEM_V3, STORY, TestSuite)
show_coverage(v3)
leaked = [w for w in ("coupon", "cart", "save10", "rs 499") if w in v3.model_dump_json().lower()]
print("Few-shot leakage:", leaked or "none")

{'tests': 40, 'criteria covered': '6/6', 'test types': '4/4', 'boundaries hit': '5/9', 'near-duplicates': 8, 'vague expected results': 1}
Missing: ['OTP used at 4:59 (just valid)', 'OTP used at 5:01 (just expired)', 'Resend tapped at 29 seconds', '4th resend is blocked']
Few-shot leakage: none


Prompt iteration, automated: Fill the gaps

In [ ]:
class TestSuiteAny(TestSuite):
    test_cases: list[TestCase] = Field(min_length=1)    # gap rounds may return just 1-2 tests

def fill_gaps(suite: TestSuite, rounds: int = 1) -> TestSuite:
    """Prompt iteration, automated: code finds what's missing, the model writes only those tests."""
    for r in range(rounds):
        missing = coverage(suite)["_missing"]
        if not missing:
            break
        system = f"""MODE: GAPS. {QA_RULES}
You will receive a user story and a list of scenarios the current test suite does NOT cover.
Write one focused test per missing scenario. Return ONLY JSON matching:
{json.dumps(TestSuite.model_json_schema(), indent=2)}"""
        user = STORY + "\n\nMissing scenarios:\n" + "\n".join(f"- {m}" for m in missing)
        extra = ask_json(system, user, TestSuiteAny)
        suite = TestSuite(story_id=suite.story_id, test_cases=suite.test_cases + extra.test_cases)
    for i, t in enumerate(suite.test_cases, 1):        # renumber so IDs stay unique
        t.id = f"TC-{i:03d}"
    return suite

final = fill_gaps(v3, 2)
show_coverage(final)

TypeError: show_coverage() takes 1 positional argument but 2 were given

Export for Jira or excel